# Pruning em CNNs — MIO-TCD (Kaggle)

**Uso opcional.** Este notebook roda o pipeline na GPU gratuita do Kaggle. Foi assim que o Leon rodou os Exp 1, 5 e 6. O Gabriel usa o próprio computador por padrão; use este notebook só se quiser ou precisar de GPU (ver `docs/KAGGLE.md`).

Requisitos: **Accelerator = GPU**, **Internet = On**, dataset `leonloureno/mio-tcd-subset-46k` adicionado em *Input* (compartilhado pelo Leon) e o secret `GH_TOKEN` (token do GitHub só com leitura deste repositório) ativado em *Add-ons → Secrets*.

Protocolo: `docs/SPEC.md`. Altere só `SEED` abaixo. Para os Exp 2–4, troque as células do Exp 1/5/6 pelo script da poda L1 e adicione o checkpoint do baseline como *Input*.

Para rodar só a poda sem treinar o baseline de novo: adicione como *Input* a saída da versão que treinou o baseline e use `RUN_BASELINE = False`.

In [ ]:
SEED = 0
RUN_BASELINE = True  # False: reaproveita o checkpoint do baseline de uma versao anterior adicionada como Input

import glob, os, shutil, subprocess
from kaggle_secrets import UserSecretsClient

REPO = "/kaggle/working/cnn-pruning-miotcd"
tok = UserSecretsClient().get_secret("GH_TOKEN")
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", f"https://{tok}@github.com/LeonLourenco/cnn-pruning-miotcd.git", REPO], check=True)
# remove o token do .git/config: a pasta /kaggle/working vira saida publica da versao
subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", "https://github.com/LeonLourenco/cnn-pruning-miotcd.git"], check=True)
del tok
os.chdir(REPO)
print(subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q torch-pruning==1.6.0

# localiza a pasta do dataset (o caminho de montagem varia entre versoes do Kaggle)
cands = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/car", recursive=True)]
assert cands, "dataset mio-tcd-subset-46k nao encontrado em /kaggle/input"
IMG_ROOT = cands[0]
print("imagens em", IMG_ROOT, "| classes:", len(os.listdir(IMG_ROOT)))

# aponta o config para o Kaggle (mudanca so nesta sessao, nao commitar)
!sed -i "s#img_root: .*#img_root: {IMG_ROOT}#" configs/default.yaml
!sed -i "s#num_workers: 2#num_workers: 4#" configs/default.yaml
!grep -n "img_root\|num_workers" configs/default.yaml

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA")
assert torch.cuda.is_available(), "ative Accelerator = GPU"

## Exp 1 — baseline denso

In [ ]:
if RUN_BASELINE:
    !python -m src.train --seed {SEED}
else:
    found = glob.glob(f"/kaggle/input/**/exp1_baseline_s{SEED}/exp1_baseline_s{SEED}.pt", recursive=True)
    assert found, "adicione como Input a saida da versao que treinou o baseline"
    shutil.copytree(os.path.dirname(found[0]), f"results/exp1_baseline_s{SEED}", dirs_exist_ok=True)
    print("baseline reaproveitado de", os.path.dirname(found[0]))

## Exp 5 e 6 — poda estruturada (máscara e física)

In [ ]:
!python scripts/run_structured.py --seed {SEED}

## Resumo e saída

`results_s{SEED}.zip` aparece na aba *Output*. Baixe e extraia em `results/` no repositório local.
`baseline_s{SEED}.zip` é o checkpoint do baseline para compartilhar com o Gabriel (D-06).

In [ ]:
!python scripts/aggregate_results.py
!cd /kaggle/working/cnn-pruning-miotcd && zip -q -r /kaggle/working/results_s{SEED}.zip results
!cd /kaggle/working/cnn-pruning-miotcd/results && zip -q -r /kaggle/working/baseline_s{SEED}.zip exp1_baseline_s{SEED}
!ls -la /kaggle/working/*.zip